# Machine Learning Notes
## Day 31: Working with Time and Date Data — Answer Key

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Feature Engineering for Datetime Columns  
> **Difficulty:** Beginner to Intermediate  

---
### Note:
FULLY WORKED SOLUTIONS for every exercise in **Day31_Working_with_Time_Date_Practice_Questions.ipynb**.

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

plt.rcParams['figure.figsize'] = (10, 4)
print('Libraries imported!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Parsing Date Strings to Datetime

In [ ]:
# ============================================================
# ANSWER 1: Parse and inspect datetime columns
# ============================================================

df = pd.DataFrame({
    'order_id':     [1, 2, 3, 4, 5],
    'order_date':   ['2024-01-15', '2024-03-22', '2024-07-04', '2024-11-30', '2024-12-25'],
    'delivery_date':['2024-01-18', '2024-03-25', '2024-07-08', '2024-12-03', '2024-12-28']
})

# 1. Before conversion
print('Dtypes BEFORE conversion:')
print(df.dtypes)

# 2. Convert both columns
df['order_date']    = pd.to_datetime(df['order_date'],    format='%Y-%m-%d')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], format='%Y-%m-%d')

# 3. After conversion
print('\nDtypes AFTER conversion:')
print(df.dtypes)

# 4. Min and max
print(f'\nEarliest order date: {df["order_date"].min()}')
print(f'Latest order date:   {df["order_date"].max()}')
print(df)

---
## Section 2: Extracting Basic Date and Time Components

In [ ]:
# ============================================================
# ANSWER 2: Extract all components from transaction timestamp
# ============================================================

df2 = pd.DataFrame({'transaction_ts': [
    '2024-01-15 08:30:00', '2024-03-22 13:45:00', '2024-07-04 23:15:00',
    '2024-11-30 06:00:00', '2024-12-25 18:30:00', '2024-08-10 12:00:00',
]})
df2['transaction_ts'] = pd.to_datetime(df2['transaction_ts'])

# Extract components
df2['year']         = df2['transaction_ts'].dt.year
df2['month']        = df2['transaction_ts'].dt.month
df2['day']          = df2['transaction_ts'].dt.day
df2['hour']         = df2['transaction_ts'].dt.hour
df2['minute']       = df2['transaction_ts'].dt.minute
df2['day_of_week']  = df2['transaction_ts'].dt.dayofweek
df2['quarter']      = df2['transaction_ts'].dt.quarter
df2['day_of_year']  = df2['transaction_ts'].dt.dayofyear
df2['month_name']   = df2['transaction_ts'].dt.month_name()
df2['day_name']     = df2['transaction_ts'].dt.day_name()

print('All extracted components:')
print(df2.to_string(index=False))

---
## Section 3: Creating Binary Flags

In [ ]:
# ============================================================
# ANSWER 3: Binary flag features
# ============================================================

df3 = pd.DataFrame({
    'timestamp': pd.date_range(start='2024-01-01', periods=14, freq='D').append(
                 pd.to_datetime(['2024-01-31', '2024-03-31', '2024-06-30']))
})

# 1. is_weekend: Saturday(5) or Sunday(6)
df3['is_weekend']     = df3['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

# 2. is_weekday: Monday(0) to Friday(4)
df3['is_weekday']     = (df3['timestamp'].dt.dayofweek < 5).astype(int)

# 3. is_month_end
df3['is_month_end']   = df3['timestamp'].dt.is_month_end.astype(int)

# 4. is_quarter_end
df3['is_quarter_end'] = df3['timestamp'].dt.is_quarter_end.astype(int)

df3['day_name'] = df3['timestamp'].dt.day_name()
print(df3.to_string(index=False))

print('\nVerification: weekends should be Sat/Sun, month_end = last day of each month')

---
## Section 4: Computing Age, Duration, and Days-Since Features

In [ ]:
# ============================================================
# ANSWER 4: Time difference features
# ============================================================

today = pd.Timestamp('2024-12-31')
df4 = pd.DataFrame({
    'CustomerID':        [1, 2, 3, 4, 5],
    'date_of_birth':     ['1990-05-15', '1985-11-22', '2000-03-08', '1975-07-30', '1995-01-01'],
    'account_open_date': ['2015-01-10', '2018-06-15', '2020-09-01', '2010-03-20', '2022-12-01'],
    'last_login_date':   ['2024-12-28', '2024-11-15', '2024-12-31', '2024-10-01', '2024-12-20'],
})

# 1. Convert all date columns
for col in ['date_of_birth', 'account_open_date', 'last_login_date']:
    df4[col] = pd.to_datetime(df4[col])

# 2. Age in complete years
df4['age_years'] = ((today - df4['date_of_birth']).dt.days // 365)

# 3. Account age in days
df4['account_age_days'] = (today - df4['account_open_date']).dt.days

# 4. Days since last login (recency)
df4['days_since_login'] = (today - df4['last_login_date']).dt.days

print('Derived time features:')
print(df4[['CustomerID', 'age_years', 'account_age_days', 'days_since_login']].to_string(index=False))

print('\nHigher days_since_login = customer has been inactive longer = churn risk signal')

---
## Section 5: Cyclical Encoding of Circular Time Features

In [ ]:
# ============================================================
# ANSWER 5: Cyclical encoding and visualisation
# ============================================================

hours = pd.DataFrame({'hour': range(24)})

# 1. Cyclical encoding for hour (period = 24)
hours['hour_sin'] = np.sin(2 * np.pi * hours['hour'] / 24)
hours['hour_cos'] = np.cos(2 * np.pi * hours['hour'] / 24)

# 2. Plot
plt.figure(figsize=(12, 4))
plt.plot(hours['hour'], hours['hour_sin'], 'b-o', markersize=5, label='hour_sin')
plt.plot(hours['hour'], hours['hour_cos'], 'r-s', markersize=5, label='hour_cos')
plt.xlabel('Hour of Day'); plt.ylabel('Encoded Value')
plt.title('Cyclical Encoding of Hour\nAmol Jagtap | amoljagtap3001@gmail.com')
plt.xticks(range(24)); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 3. Verify adjacency
print('Hour 0  sin/cos:', hours.loc[0,  ['hour_sin','hour_cos']].round(3).values)
print('Hour 23 sin/cos:', hours.loc[23, ['hour_sin','hour_cos']].round(3).values)
print('Hour 12 sin/cos:', hours.loc[12, ['hour_sin','hour_cos']].round(3).values)

dist_0_23 = np.sqrt((hours.loc[0,'hour_sin']-hours.loc[23,'hour_sin'])**2 +
                     (hours.loc[0,'hour_cos']-hours.loc[23,'hour_cos'])**2)
dist_0_12 = np.sqrt((hours.loc[0,'hour_sin']-hours.loc[12,'hour_sin'])**2 +
                     (hours.loc[0,'hour_cos']-hours.loc[12,'hour_cos'])**2)
print(f'\nDistance (hour 0 to 23): {dist_0_23:.3f}  ← SMALL (they are adjacent!)')
print(f'Distance (hour 0 to 12): {dist_0_12:.3f}  ← LARGE (they are opposites!)')

# 4. Also for day of week (period=7) and month (period=12)
dow_df = pd.DataFrame({'day': range(7)})
dow_df['dow_sin'] = np.sin(2 * np.pi * dow_df['day'] / 7)
dow_df['dow_cos'] = np.cos(2 * np.pi * dow_df['day'] / 7)

month_df = pd.DataFrame({'month': range(1, 13)})
month_df['month_sin'] = np.sin(2 * np.pi * month_df['month'] / 12)
month_df['month_cos'] = np.cos(2 * np.pi * month_df['month'] / 12)

print('\nDay of week cyclical encoding:')
print(dow_df.round(3).to_string(index=False))

---
## Section 6: Handling Different Date Formats and NaT Values

In [ ]:
# ============================================================
# ANSWER 6: Parsing messy dates
# ============================================================

messy_dates = pd.DataFrame({'date_str': [
    '15-01-2024', '03/22/2024', 'NOT_A_DATE', '2024-07-04', '', '25 Dec 2024',
]})

# 1. Parse with errors='coerce' — unparseable → NaT
messy_dates['parsed'] = pd.to_datetime(messy_dates['date_str'], errors='coerce')

print('After parsing (errors=coerce):')
print(messy_dates)

# 2. Count NaT values
nat_count = messy_dates['parsed'].isna().sum()
print(f'\nNumber of NaT values: {nat_count}')

# 3. Fill NaT with a default date
messy_dates['parsed_filled'] = messy_dates['parsed'].fillna(pd.Timestamp('2024-01-01'))
print('\nAfter filling NaT with 2024-01-01:')
print(messy_dates[['date_str', 'parsed', 'parsed_filled']])

# 4. Unix timestamps
unix_df = pd.DataFrame({'unix_ts': [1704067200, 1710720000, 1719014400]})
unix_df['readable_date'] = pd.to_datetime(unix_df['unix_ts'], unit='s')
print('\nUnix timestamps converted to readable dates:')
print(unix_df)

---
## Section 7: Time Zone Conversion

In [ ]:
# ============================================================
# ANSWER 7: Standardise timestamps to IST
# ============================================================

df7 = pd.DataFrame({
    'event':     ['Login', 'Purchase', 'Logout'],
    'timestamp': ['2024-07-04 10:00:00', '2024-07-04 14:30:00', '2024-07-04 15:00:00']
})
df7['timestamp'] = pd.to_datetime(df7['timestamp'])

# 1. Localize to UTC
df7['ts_utc'] = df7['timestamp'].dt.tz_localize('UTC')

# 2. Convert to IST (UTC + 5:30)
df7['ts_ist'] = df7['ts_utc'].dt.tz_convert('Asia/Kolkata')

# 3. Extract hour in IST
df7['hour_ist'] = df7['ts_ist'].dt.hour

print('UTC vs IST comparison:')
print(df7[['event', 'ts_utc', 'ts_ist', 'hour_ist']].to_string(index=False))

print('\nIST = UTC + 5 hours 30 minutes')
print(f'UTC 10:00 → IST {10 + 5}:{30:02d} → IST hour = 15')
print('Standardising timezone is critical before using hour-of-day as a feature.')

---
## Section 8: Mini End-to-End Datetime Feature Engineering Pipeline

In [ ]:
# ============================================================
# ANSWER 8: Full datetime feature engineering pipeline
# ============================================================

np.random.seed(42)
n = 300
ref_date = pd.Timestamp('2024-12-31')

raw = pd.DataFrame({
    'transaction_ts': pd.date_range('2023-01-01', periods=n, freq='25H'),
    'account_open':   pd.date_range('2015-01-01', periods=n, freq='10D'),
    'amount':         np.random.uniform(100, 10000, n),
})
raw['fraud'] = (
    (raw['transaction_ts'].dt.dayofweek >= 5) |
    (raw['transaction_ts'].dt.hour.isin([0,1,2,3]))
).astype(int)

df_eng = raw.copy()

# Step 1: Extract features from transaction_ts
df_eng['year']          = df_eng['transaction_ts'].dt.year
df_eng['month']         = df_eng['transaction_ts'].dt.month
df_eng['day']           = df_eng['transaction_ts'].dt.day
df_eng['hour']          = df_eng['transaction_ts'].dt.hour
df_eng['day_of_week']   = df_eng['transaction_ts'].dt.dayofweek
df_eng['is_weekend']    = df_eng['transaction_ts'].dt.dayofweek.isin([5,6]).astype(int)
df_eng['is_rush_hour']  = df_eng['transaction_ts'].dt.hour.isin([8,9,17,18]).astype(int)

# Cyclical encoding
df_eng['month_sin']     = np.sin(2 * np.pi * df_eng['transaction_ts'].dt.month / 12)
df_eng['month_cos']     = np.cos(2 * np.pi * df_eng['transaction_ts'].dt.month / 12)
df_eng['hour_sin']      = np.sin(2 * np.pi * df_eng['transaction_ts'].dt.hour / 24)
df_eng['hour_cos']      = np.cos(2 * np.pi * df_eng['transaction_ts'].dt.hour / 24)

# Step 2: Account age in days
df_eng['account_age_days'] = (ref_date - df_eng['account_open']).dt.days

# Step 3: Drop original datetime columns
df_eng.drop(columns=['transaction_ts', 'account_open'], inplace=True)

print('Engineered features:', df_eng.columns.tolist())
print('Shape:', df_eng.shape)

# Steps 4 & 5: Split
X = df_eng.drop(columns=['fraud'])
y = df_eng['fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Step 6: Train RandomForest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f'\nTest accuracy: {rf.score(X_test, y_test):.4f}')

# Step 7: Top 5 feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\nTop 5 most important features:')
print(importances.head(5).round(4))
print('\nis_weekend and hour-based features should appear near the top')
print('since they are directly encoded in the fraud target.')

print('\nAmol Jagtap | amoljagtap3001@gmail.com')

---
## Summary & Quick Revision

| Concept | What You Learned |
|---|---|
| Parse datetime | `pd.to_datetime(col, format='...')` converts string → datetime64 |
| .dt accessor | Extracts year, month, day, hour, dayofweek, quarter etc. |
| is_weekend | `dt.dayofweek.isin([5,6]).astype(int)` |
| Age / Duration | `(today - df['date']).dt.days` |
| Cyclical encoding | `sin(2π × value / period)` and `cos(2π × value / period)` |
| Why sin AND cos? | Single sin is ambiguous — together they uniquely encode a circle position |
| NaT | Not a Time — use `errors='coerce'`, then handle like NaN |
| Time zones | `tz_localize()` → attach; `tz_convert()` → change |
| Pipeline | Extract datetime features BEFORE Pipeline; drop raw datetime column |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 31 — Working with Time and Date Data